## LSTM (Long Short Term Memory):- 
Long Short term memory is a type of the sequential model, that takes one timestep at a time in order process it and produced two state, cell state and the hidden state.

Cell staate c(t) is the long term memory, that contains the data which are important for long term. It contains the understanding that the model has understood so far that is important in future timestep and they might depend upon but we dont know which and what will be important.

Hidden state h(t) is the short term memory, that contains the contextual understanding/summary or the snapshot of the lstm model throughout the process till the previous timestep. It does hold the information that is important and crucial for now.

It makes the LSTM, different from the RNN because, RNN stores everything in one state due to which the information that we will be loosing is the long term information, which removes the long term dependency/quality information and it causes the lack of prediciton quality.

Whereas, in LSTM there are two state, cell state only holds the information that are found in early timestep as well as the information that can be crucial for future timesteps. And the short term memory s like the hidden state of rnn, as it holds the contextual understanding till the previous timestep. And the short term memory contains the information crucial in current scenario. And this provides us the ability to play with the LTM and STM like, we can decide what should we forget or is no longer important, what should we consider the information that can be important in future and what are the information we neet to add to STM from LTM which can be crucial for next time step.

Provides us control to what to forget, what to remember and what to consider right now.

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.optim as optim
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import nltk
from nltk.tokenize import word_tokenize
from collections import Counter

In [2]:
nltk.download("punkt_tab")
nltk.download("punkt")

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Nitro\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Nitro\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

### we cannot load all the text from the text.txt file all at once because of the RAM gettig full, so we will use generator to produce desired number of text at a time

In [4]:
def file(file_name):
    with open(file_name, "r") as f:
            for line in f:
                yield line

In [5]:
def tokens_generator(f):
    tokens = []
    try:
        while True:
            text = next(f)
            tokens.extend(word_tokenize(text.lower()))
    except StopIteration:
        return tokens

In [6]:
def voccabulary(file_name):
    f = file(file_name)
    tokens = tokens_generator(f)
    voccab = {
        '<UNK>' : 0,
        '<PAD>' : 1,
    }
    for token in Counter(tokens):
        voccab[token] = len(voccab)
    return voccab

In [7]:
voccab = voccabulary("smaller.txt")

In [8]:
len(voccab)

10473

In [9]:
def text_to_indices(voccab):
    f = file("smaller.txt")
    dataset = []
    max_len = 0
    # extract text from file
    try:
        while True:
            # tokenize
            indices = []
            text = next(f)
            tokens = word_tokenize(text.lower())
            if max_len < len(tokens):
                max_len = len(tokens)
            # replace with respective integer from voccab
            for token in tokens:
                indices.append(voccab[token])
            # preparing the dataset from indices
            for element in range(len(indices) - 1):
                dataset.append(indices[ : element + 2])
    except StopIteration:
        return  max_len, dataset

In [10]:
indices = text_to_indices(voccab)

### we need to create the dataset (inputs) for the LSTM model
As we are trying to make the next word predction so, in a sequence there can be different size of sentence so we need to perfrom the padding to make them of the same size as well.

In [11]:
def dataset_preparatioin(max_length, indices):
    dataset = []
    for i in indices:
        dataset.append([1]*(max_length - len(i)) + i)
    return dataset

In [12]:
dataset = dataset_preparatioin(indices[0], indices[1])

### Now, lets create the dataset and data loader for input and expected output from the model

In [13]:
class CustomDataSet(Dataset):
    def __init__(self, texts):
        self.features = torch.tensor(texts)[:, :-1].to(device)
        self.target = torch.tensor(texts)[:, -1].to(device)
    
    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, index):
        return (self.features[index], self.target[index])

In [14]:
data = CustomDataSet(dataset)

In [15]:
data.__len__()

188210

In [16]:
data.__getitem__(0)

(tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2],
        device='cuda:0'),
 tensor(3, device='cuda:0'))

In [17]:
dataloader = DataLoader(data, batch_size = 128, shuffle = True)

In [18]:
class NextWordPredictor(nn.Module):

    def __init__(self, voccab):
        super().__init__()

        self.embedding = nn.Embedding(num_embeddings = len(voccab), embedding_dim = 100, padding_idx = 1)
        self.lstm = nn.LSTM(input_size = 100, hidden_size = 256, num_layers = 3, batch_first = True)
        self.linear = nn.Linear(256, len(voccab))
    
    def forward(self, features):
        embed = self.embedding(features)
        lstm = self.lstm(embed)[1][0][-1]
        linear = self.linear(lstm)
        return linear

#### Why do we use the multiple layer in LSTM?
- As in the case of ANN, during the image classification we need multiple hidden layers because, layer by layer the basic/primitive feature is turned to complex pattern that can classifiy the image.
- from edges --> structure --> objects --> complex objects --> shape --> item.
- it starts to see the small part and most basic part of image that is edge and then uses the edges to discover and learn more.

#### Similarly, in the case of the LSTM, 
- when we add multiple number of layers in LSTM then the output of one LSTM goes to the input of the other one and so on.
- Later on training, it slowly first understand the meaning of the word, grammar, pharases and then the whole sentence.
- At the end of the training, language understanding is developed in the model and ability to really understand the meaning and goal of the text.
- Its the same as ANN where, it learns the primitive features and the combines them in each layer to make them the complex one able to classify or provide the numberical value.
- In LSTM, multiple hidden layers allow model to learn the data from base that it its syllable, grammar, phrases. How does different word influences the sentence. It then starts to learn the sentiment and meaninig of the whole sentence when processing step by step.

internal hidden state of each of the timestep of one layer is input to the following layer stacked and so on. As the hidden state (understanding and summary of all the timestep processed by the model till that timestep) is given, and next layer uses that state for further processing, the primitive understading level ups layer by layer.

At the end, we get the hidden state of each of the layer at last timestep and we just use the last layer hidden state. As it will contain the overall summary or understadning and whats crucial right now. Unlike the cell state that is an store where all the important topics, understading are stored if may be required in future.

The hidden state is the LSTM's current contextual understanding of everything it has seen so far. At each timestep, this understanding is updated using the current input and the updated memory in the same timestep(cell state)

#### Stacking the models is also done in LLM, where we stack the transformers above one another.

In [19]:
model = NextWordPredictor(voccab)
model.to(device=device)

NextWordPredictor(
  (embedding): Embedding(10473, 100, padding_idx=1)
  (lstm): LSTM(100, 256, num_layers=3, batch_first=True)
  (linear): Linear(in_features=256, out_features=10473, bias=True)
)

In [20]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = 0.001)
epochs = 50

In [21]:
for epoch in range(epochs):
    net_loss = 0
    for features, target in dataloader:
        output = model(features)
        loss = loss_fn(output, target)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        net_loss = net_loss + loss.item()
    average_loss = net_loss/len(dataloader)
    print("epoch ===>", epoch + 1, "loss ===>", average_loss)    

epoch ===> 1 loss ===> 6.350564837050551
epoch ===> 2 loss ===> 5.480201968848503
epoch ===> 3 loss ===> 5.0860889368037805
epoch ===> 4 loss ===> 4.810650592430524
epoch ===> 5 loss ===> 4.576579403666555
epoch ===> 6 loss ===> 4.362984832819427
epoch ===> 7 loss ===> 4.162877145387299
epoch ===> 8 loss ===> 3.972908636082767
epoch ===> 9 loss ===> 3.794702052907989
epoch ===> 10 loss ===> 3.6309121666434345
epoch ===> 11 loss ===> 3.4796675848037024
epoch ===> 12 loss ===> 3.340051795575669
epoch ===> 13 loss ===> 3.2109394726665714
epoch ===> 14 loss ===> 3.0896671388847823
epoch ===> 15 loss ===> 2.975260309748387
epoch ===> 16 loss ===> 2.868199108734494
epoch ===> 17 loss ===> 2.7649622537261527
epoch ===> 18 loss ===> 2.6671875657717603
epoch ===> 19 loss ===> 2.575580941504317
epoch ===> 20 loss ===> 2.486020200111524
epoch ===> 21 loss ===> 2.4027479549073427
epoch ===> 22 loss ===> 2.320097016850594
epoch ===> 23 loss ===> 2.2415319503690823
epoch ===> 24 loss ===> 2.16583361

### Saving the model
We know a simple fact that, everything that we have trained all the weights and biases (parameters) are stored in RAM and when we close the application, whole memory gets released and we loose what e trained taking so much time.
So, we save the model itself, which gets saved with all the parametrs, 

i.e. model.parameters() as pth, so that we can load all the wieghts back and use it  without training our model each time we open our code

In [21]:
for i in model.named_parameters():
    print(i)
    print("=" * 100)

('embedding.weight', Parameter containing:
tensor([[ 0.5607, -0.8287, -1.3672,  ...,  1.0347,  1.0429,  0.1260],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [-0.1819,  0.1223,  0.6913,  ...,  0.2663, -0.4139, -0.2849],
        ...,
        [ 0.5113,  0.3292, -0.4841,  ..., -2.6937, -0.1241,  0.0429],
        [ 1.4497, -0.4817, -0.3219,  ...,  0.2784,  0.1649, -1.9431],
        [ 1.8720, -1.1305, -0.5784,  ...,  1.3802,  2.3074,  1.0482]],
       device='cuda:0', requires_grad=True))
('lstm.weight_ih_l0', Parameter containing:
tensor([[ 0.0119,  0.0133,  0.0450,  ...,  0.0290, -0.0374,  0.0551],
        [-0.0019,  0.0427,  0.0191,  ..., -0.0421, -0.0578,  0.0042],
        [ 0.0580, -0.0423,  0.0087,  ..., -0.0278, -0.0533, -0.0046],
        ...,
        [-0.0496,  0.0413,  0.0330,  ..., -0.0311, -0.0342, -0.0353],
        [-0.0466,  0.0252,  0.0064,  ...,  0.0287, -0.0237, -0.0624],
        [ 0.0463, -0.0037, -0.0112,  ..., -0.0380,  0.0024,  0.0624]],


After our model gets trained, all the parameters above we get are optimized wieghts and baises. Thye carry the learning and knowledge to understand word, phrases, theri emotion, meaning, impact etc. which we cannot loose.

So,

In [22]:
torch.save(model.state_dict(), "model.pth")

Now, the models are stored in secondary storage, we can import the model and all the optimized weights + biases are ready to be used

In [24]:
# this are the list of different parameters in our system 
for i in model.state_dict():
    print(i)

embedding.weight
lstm.weight_ih_l0
lstm.weight_hh_l0
lstm.bias_ih_l0
lstm.bias_hh_l0
lstm.weight_ih_l1
lstm.weight_hh_l1
lstm.bias_ih_l1
lstm.bias_hh_l1
lstm.weight_ih_l2
lstm.weight_hh_l2
lstm.bias_ih_l2
lstm.bias_hh_l2
linear.weight
linear.bias


Now, for loading the model that we saved, we need to create the model object, that will contain the random weights and then assign all the weights to it which was stored.

state_dict() will be assigned to the model

In [ ]:
model.load_state_dict(torch.load("model.pth"))

In [ ]:
# testing the accuracy